<a href="https://colab.research.google.com/github/0ai-Cyberviser/0ai/blob/main/Hancock_Colab_Finetune_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# 🚨 HARDWARE REQUIREMENT: T4 GPU
To use Unsloth for fast fine-tuning, you must enable a GPU:
1. **Runtime** > **Change runtime type**
2. **Hardware accelerator** > **T4 GPU**
3. **Save**

*If you do not see T4 GPU, you may need to clear your active sessions or use a different Google account.*
```

# 🔐 HancockForge — CyberViser / 0AI
**Mistral 7B → Cybersecurity Specialist via LoRA (v0.3.1-dev)**

| Feature | Status | Tech Stack |
|:---|:---|:---|
| **Model** | Mistral-7B-Instruct-v0.3 | Unsloth / 4-bit |
| **Orchestration** | LangGraph (Planned) | Python / FastAPI |
| **Data** | CISA KEV + Atomic + GHSA | hancock_pipeline.py |
| **Deployment** | GGUF / Ollama / NIM | Docker / K8s |

> **⚠️ CRITICAL:** Ensure **Runtime → Change runtime type → T4 GPU** is enabled.

In [5]:
# @title 1️⃣  Install Dependencies
# Pin versions for compatibility
!pip install -q "unsloth[colab-new]" \
    "trl>=0.8.6,<0.10" \
    "transformers>=4.40,<4.46" \
    "datasets>=2.18" \
    "peft" "huggingface_hub" "accelerate"
import importlib, pkg_resources
for pkg in ['unsloth','trl','transformers','datasets','peft']:
    v = pkg_resources.get_distribution(pkg).version
    print(f'  {pkg}: {v}')
print('✅ Deps installed')

  unsloth: 2024.10.7
  trl: 0.9.6
  transformers: 4.44.2
  datasets: 4.0.0
  peft: 0.18.1
✅ Deps installed


In [6]:
# @title 2️⃣  Clone Hancock Repo
import os
!git clone https://github.com/cyberviser/Hancock.git /content/Hancock
os.chdir('/content/Hancock')
print('✅ Repo cloned')

fatal: destination path '/content/Hancock' already exists and is not an empty directory.
✅ Repo cloned


In [10]:
# @title 3️⃣  Check GPU & Environment
import torch
import sys

if not torch.cuda.is_available():
    print("❌ ERROR: CUDA not found. Please go to 'Runtime' -> 'Change runtime type' and select 'T4 GPU'.")
    # Do not raise AssertionError here to allow the user to see the UI prompt cell below if needed
else:
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu} | VRAM: {vram:.1f} GB')
    print(f'✅ Python: {sys.version.split()[0]}')


❌ ERROR: CUDA not found. Please go to 'Runtime' -> 'Change runtime type' and select 'T4 GPU'.


In [9]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


GridBox(children=(Dropdown(layout=Layout(width='auto'), options=('google/gemini-2.5-flash', 'google/gemini-2.5…

```markdown
## 🔄 LangGraph Agentic Orchestration
Hancock is evolving from a single-prompt model to a multi-agent system. The loop below implements the **Planner-Executor-Critic** pattern:

1.  **Planner**: Deconstructs the user query into specific technical tasks (e.g., Nmap scan -> CVE research).
2.  **Executor**: The fine-tuned Mistral-7B generates the specific commands or analysis.
3.  **Ethical Critic**: Validates the output against the Verbatim Pentest Mode system prompt (authorized scope check).
4.  **Router**: Determines if the goal is met or if a loop back to Planning is required.
```

In [12]:
from typing import Annotated, List, TypedDict
import operator
from langgraph.graph import StateGraph, END

# Define the shared state for the Hancock Agents
class HancockState(TypedDict):
    task: str
    plan: List[str]
    findings: List[str]
    critique: str
    authorized_scope: bool
    iteration_count: int

def planning_node(state: HancockState):
    print("--- AGENT: PLANNER ---")
    # In production, this calls Mistral with the 'CISO/Planner' prompt
    return {"plan": ["Enumerate services", "Identify CVEs", "Draft Report"], "iteration_count": state.get('iteration_count', 0) + 1}

def execution_node(state: HancockState):
    print("--- AGENT: EXECUTOR ---")
    # This uses the fine-tuned Hancock model to generate technical content
    return {"findings": state.get('findings', []) + ["Found OpenSSH 8.2p1 on port 22"]}

def safety_critic_node(state: HancockState):
    print("--- AGENT: ETHICAL CRITIC ---")
    # Validates technical precision and authorization compliance
    return {"authorized_scope": True, "critique": "Logic verified. No out-of-scope actions detected."}

def should_continue(state: HancockState):
    if state.get('iteration_count', 0) >= 3 or state.get('authorized_scope') == False:
        return "end"
    return "continue"

# Build the Graph
workflow = StateGraph(HancockState)

workflow.add_node("planner", planning_node)
workflow.add_node("executor", execution_node)
workflow.add_node("critic", safety_critic_node)

workflow.set_entry_point("planner")
workflow.add_edge("planner", "executor")
workflow.add_edge("executor", "critic")

workflow.add_conditional_edges(
    "critic",
    should_continue,
    {
        "continue": "planner",
        "end": END
    }
)

hancock_app = workflow.compile()
print("✅ Hancock LangGraph Orchestrator Initialized")

✅ Hancock LangGraph Orchestrator Initialized


### 🐳 Secure Execution Sandbox (DinD)
To prevent Hancock from damaging the host system, we wrap technical commands in a Docker container. This node takes the output from the `executor` and runs it in a restricted shell.

In [13]:
import subprocess

def sandbox_execution_node(state: HancockState):
    print("--- AGENT: SANDBOX EXECUTOR ---")

    # In a real scenario, we would pull the 'hancock-tool-suite' image
    # Here we simulate the safe execution of the last finding/command
    last_finding = state['findings'][-1]

    # Example command construction (Simulated)
    safe_command = f"echo 'Running in Sandbox: {last_finding}'"

    try:
        # Simulated execution via subprocess (would be 'docker run' in prod)
        result = subprocess.check_output(safe_command, shell=True, text=True)
        execution_log = f"[SANDBOX SUCCESS] {result.strip()}"
    except Exception as e:
        execution_log = f"[SANDBOX ERROR] {str(e)}"

    return {"findings": state['findings'] + [execution_log]}

# Re-compile workflow with Sandbox integration
workflow_v2 = StateGraph(HancockState)
workflow_v2.add_node("planner", planning_node)
workflow_v2.add_node("executor", execution_node)
workflow_v2.add_node("sandbox", sandbox_execution_node)
workflow_v2.add_node("critic", safety_critic_node)

workflow_v2.set_entry_point("planner")
workflow_v2.add_edge("planner", "executor")
workflow_v2.add_edge("executor", "sandbox")
workflow_v2.add_edge("sandbox", "critic")

workflow_v2.add_conditional_edges(
    "critic",
    should_continue,
    {"continue": "planner", "end": END}
)

hancock_sandbox_app = workflow_v2.compile()
print("✅ Hancock Agentic Sandbox (v2) Initialized")

✅ Hancock Agentic Sandbox (v2) Initialized


In [14]:
# @title 4​​  Load Training Data
import json
from pathlib import Path

# Prioritize the specific v1 pentest dataset requested
dataset_path = Path('/content/Hancock/data/hancock_pentest_v1.jsonl')

# Fallback logic if v1 isn't found, try to locate v3 or generate
if not dataset_path.exists():
    print(f'☑️ v1 not found at {dataset_path}, checking alternatives...')
    dataset_path = Path('/content/Hancock/data/hancock_v3.jsonl')

if not dataset_path.exists():
    print('Generating dataset via pipeline...')
    !python hancock_pipeline.py --phase 3
    dataset_path = Path('/content/Hancock/data/hancock_v3.jsonl')

if dataset_path.exists():
    lines = dataset_path.read_text().strip().splitlines()
    data  = [json.loads(l) for l in lines]
    print(f'✅ Loaded {len(data):,} training samples from {dataset_path.name}')
    print('Sample Message structure verified.')
else:
    print('❌ ERROR: Training data not found. Please ensure the Hancock repo is cloned and data is generated.')

✅ Loaded 1,318 training samples from hancock_pentest_v1.jsonl
Sample Message structure verified.


In [15]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'mistralai/Mistral-7B-Instruct-v0.3',
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # LoRA Rank optimized for technical specialization
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha = 32,
    lora_dropout = 0.05, # Recommended for small-medium datasets
    bias = 'none',
    use_gradient_checkpointing = 'unsloth', # Drastically reduces VRAM usage
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

print(f'✅ Model loaded and LoRA adapters applied.')
print(f'✅ Trainable params: {model.num_parameters(only_trainable=True):,}')

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
# @title 6️⃣  Format Dataset
from datasets import Dataset

texts = [
    tokenizer.apply_chat_template(s['messages'], tokenize=False, add_generation_prompt=False)
    for s in data
]
ds = Dataset.from_dict({'text': texts}).train_test_split(test_size=0.05, seed=42)
print(f'Train: {len(ds["train"]):,} | Eval: {len(ds["test"]):,}')
print('\nSample formatted text:')
print(texts[0][:400])

In [ ]:
# @title 7️⃣  Train (~45 min on T4)
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    per_device_train_batch_size  = 2,
    gradient_accumulation_steps  = 4,
    warmup_ratio                 = 0.05,
    num_train_epochs             = 3,
    learning_rate                = 2e-4,
    fp16                         = not torch.cuda.is_bf16_supported(),
    bf16                         = torch.cuda.is_bf16_supported(),
    logging_steps                = 20,
    eval_strategy                = 'steps',  # transformers>=4.45
    eval_steps                   = 100,
    save_strategy                = 'steps',
    save_steps                   = 200,
    save_total_limit             = 2,
    output_dir                   = '/content/hancock_checkpoints',
    report_to                    = 'none',
    optim                        = 'adamw_8bit',
    weight_decay                 = 0.01,
    lr_scheduler_type            = 'cosine',
    seed                         = 42,
)

trainer = SFTTrainer(
    model               = model,
    tokenizer           = tokenizer,
    train_dataset       = ds['train'],
    eval_dataset        = ds['test'],
    dataset_text_field  = 'text',
    max_seq_length      = 2048,
    packing             = True,
    args                = training_args,
)

# Show initial memory usage
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
used    = torch.cuda.memory_allocated(0) / 1e9
print(f'VRAM: {used:.1f}/{gpu_mem:.1f} GB used before training')

result = trainer.train()
print(f'\n✅ Done! Final loss: {result.training_loss:.4f}')
print(f'Steps: {result.global_step} | Samples/sec: {result.training_loss:.4f}')

In [ ]:
# @title 8️⃣  Save LoRA + GGUF Q4
import os

# Always save LoRA adapters (fast, always works)
model.save_pretrained('/content/hancock_lora')
tokenizer.save_pretrained('/content/hancock_lora')
print('✅ LoRA adapters saved → /content/hancock_lora')
print(f'   Size: {sum(os.path.getsize(os.path.join("/content/hancock_lora",f)) for f in os.listdir("/content/hancock_lora")) / 1e6:.1f} MB')

# Export GGUF Q4_K_M (requires ~10 min + llama.cpp build)
try:
    model.save_pretrained_gguf('/content/hancock_gguf', tokenizer,
                               quantization_method='q4_k_m')
    print('✅ GGUF Q4_K_M saved → /content/hancock_gguf')
except Exception as e:
    print(f'⚠️  GGUF export failed (non-fatal): {e}')
    print('   LoRA adapters are saved — use them directly or convert later.')
    print('   Convert offline: python -m llama_cpp.convert ...')

In [ ]:
# @title 9️⃣  Test the Fine-Tuned Model
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

messages = [
    {'role': 'system', 'content': 'You are Hancock, an elite cybersecurity AI by CyberViser.'},
    {'role': 'user', 'content': 'Explain CVE-2021-44228 Log4Shell and how to detect it in Splunk.'},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
).to('cuda')

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs, max_new_tokens=512,
        use_cache=True, temperature=0.7, do_sample=True,
    )

# Decode only the generated tokens (not the prompt)
response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print('Hancock says:')
print('=' * 60)
print(response)

In [ ]:
# @title 🔟  Push to HuggingFace Hub (optional)
HF_TOKEN = ''  # @param {type:'string'}
if HF_TOKEN:
    model.push_to_hub('cyberviser/hancock-mistral-7b', token=HF_TOKEN)
    tokenizer.push_to_hub('cyberviser/hancock-mistral-7b', token=HF_TOKEN)
    print('✅ Pushed to https://huggingface.co/cyberviser/hancock-mistral-7b')
else:
    print('Skipped — add your HF_TOKEN to push')

In [ ]:
# @title 1️⃣1️⃣  Download GGUF to local (for Ollama)
from google.colab import files
import os
for f in os.listdir('/content/hancock_gguf'):
    if f.endswith('.gguf'):
        print(f'Downloading {f}...')
        files.download(f'/content/hancock_gguf/{f}')